In [89]:
import numpy as np
np.set_printoptions(suppress=True, precision=4)

In [90]:
def check_diag_dominant(matrix):
    sum_off_diagonal = np.sum(np.abs(matrix), axis=1) - np.abs(np.diag(matrix))
    diag = np.diag(matrix)
    compare = (sum_off_diagonal > diag).astype(int)

    print("Сума недіагональних:", sum_off_diagonal)
    print("Діагональні:", diag)
    print("Порушує діагональну перевагу (1 - так, 0 - ні):", compare)

In [91]:
A = np.array([
    [2.12, 0.42, 1.34, 0.88],
    [0.42, 3.95, 1.87, 0.43],
    [1.34, 1.87, 2.98, 0.46],
    [0.88, 0.43, 0.46, 4.44]
])

In [92]:
check_diag_dominant(A)

Сума недіагональних: [2.64 2.72 3.67 1.77]
Діагональні: [2.12 3.95 2.98 4.44]
Порушує діагональну перевагу (1 - так, 0 - ні): [1 0 1 0]


In [93]:
A = np.array([
    [2.12, 0.42, 1.34, 0.88],
    [0.42, 3.95, 1.87, 0.43],
    [1.34, 1.87, 2.98, 0.46],
    [0.88, 0.43, 0.46, 4.44]
])
b = np.array([11.172, 0.115, 0.009, 9.349])

A_1 = A.copy()
b_1 = b.copy()

A_1[0] = A_1[0] + (-1/2) * A_1[2]
b_1[0] = b_1[0] + (-1/2) * b_1[2]

print(A_1)
print(b_1)

[[ 1.45  -0.515 -0.15   0.65 ]
 [ 0.42   3.95   1.87   0.43 ]
 [ 1.34   1.87   2.98   0.46 ]
 [ 0.88   0.43   0.46   4.44 ]]
[11.1675  0.115   0.009   9.349 ]


In [94]:
check_diag_dominant(A_1)

Сума недіагональних: [1.315 2.72  3.67  1.77 ]
Діагональні: [1.45 3.95 2.98 4.44]
Порушує діагональну перевагу (1 - так, 0 - ні): [0 0 1 0]


In [95]:
A_1 = np.array([[ 1.45, -0.515, -0.15, 0.65],
 [0.42, 3.95, 1.87, 0.43],
 [1.34, 1.87, 2.98, 0.46],
 [0.88, 0.43, 0.46, 4.44]])


In [96]:

A_2 = A_1.copy()
b_2 = b_1.copy()

A_2[2] = A_2[2] + (-1/2) * A_2[0]
b_2[2] = b_2[2] + (-1/2) * b_1[0]

print(A_2)
print(b_2)

[[ 1.45   -0.515  -0.15    0.65  ]
 [ 0.42    3.95    1.87    0.43  ]
 [ 0.615   2.1275  3.055   0.135 ]
 [ 0.88    0.43    0.46    4.44  ]]
[11.1675  0.115  -5.5747  9.349 ]


In [97]:
check_diag_dominant(A_2)

Сума недіагональних: [1.315  2.72   2.8775 1.77  ]
Діагональні: [1.45  3.95  3.055 4.44 ]
Порушує діагональну перевагу (1 - так, 0 - ні): [0 0 0 0]


In [98]:
M = A_2

In [106]:
A_2, b_2

(array([[ 1.45  , -0.515 , -0.15  ,  0.65  ],
        [ 0.42  ,  3.95  ,  1.87  ,  0.43  ],
        [ 0.615 ,  2.1275,  3.055 ,  0.135 ],
        [ 0.88  ,  0.43  ,  0.46  ,  4.44  ]]),
 array([11.1675,  0.115 , -5.5747,  9.349 ]))

In [99]:
def find_C_and_d(A, b):
    C = np.zeros_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[0]):
            if i != j:
                C[i, j] = -A[i, j] / A[i, i]
    d = b / A.diagonal()
    return C, d

In [100]:
def simple_iteration(A, b, eps=1e-4, max_iter=100):
    C, d = find_C_and_d(A, b)
    q = np.max(np.sum(np.abs(C), axis=1))  # норма по рядках

    if q >= 1:
        print(f"Алгоритм не збіжний (q = {q:.4f} ≥ 1)")
        return None

    x_prev = np.zeros_like(b, dtype=float)

    for k in range(1, max_iter + 1):
        x_new = C @ x_prev + d

        delta = np.max(np.abs(x_new - x_prev))
        criterion = delta / (1 - q)

        # Вектор нев’язки
        residual = np.abs(b - A @ x_new)

        print(f"Ітерація {k}: x = {np.round(x_new, 6)}, критерій = {criterion:.6e}, r = {np.round(residual, 6)}")

        if criterion < eps:
            print(f"\nРозв’язок знайдено за {k} ітерацій:\n x = {np.round(x_new, 6)}")
            return x_new

        x_prev = x_new

    print("Досягнуто максимум ітерацій без збіжності.")
    return x_prev


In [104]:
res = simple_iteration(A_2, b_2)

Ітерація 1: x = [ 7.7017  0.0291 -1.8248  2.1056], критерій = 1.325564e+02, r = [1.6274 0.7278 5.0828 5.9506]
Ітерація 2: x = [ 6.5794 -0.1551 -3.4885  0.7654], критерій = 2.863527e+01, r = [0.5267 4.1589 1.2632 1.8322]
Ітерація 3: x = [ 6.9426  0.8978 -3.0751  1.1781], критерій = 1.812148e+01, r = [0.336  1.1032 2.5191 0.9626]
Ітерація 4: x = [ 7.1744  0.6185 -3.8997  0.9613], критерій = 1.419220e+01, r = [0.1266 1.5379 0.4809 0.2955]
Ітерація 5: x = [ 7.0871  1.0078 -3.7422  1.0278], критерій = 6.700937e+00, r = [0.1809 0.2863 0.7836 0.163 ]
Ітерація 6: x = [ 7.2118  0.9353 -3.9987  0.9911], критерій = 4.414623e+00, r = [0.0519 0.443  0.0825 0.0394]
Ітерація 7: x = [ 7.176   1.0475 -3.9717  1.    ], критерій = 1.930471e+00, r = [0.056  0.0392 0.2178 0.0291]
Ітерація 8: x = [ 7.2146  1.0375 -4.043   0.9934], критерій = 1.227005e+00, r = [0.0115 0.1199 0.0017 0.0031]
Ітерація 9: x = [ 7.2067  1.0679 -4.0436  0.9941], критерій = 5.224389e-01, r = [0.0151 0.0041 0.0598 0.0058]
Ітерація 1

In [102]:
7.22006
1.08331
-4.07652
0.992054

0.992054